In [3]:
print("Environment loaded")

Environment loaded


In [16]:
#Imports & Environment Setup
import os
import sys
import time
from pathlib import Path
from dotenv import load_dotenv

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Load environment
load_dotenv(project_root / ".env")

# Check for API key (OpenRouter preferred, OpenAI as fallback)
# openrouter_key = os.getenv("OPENROUTER_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

if not groq_key:
    raise EnvironmentError(
        "❌ No API key found!\n"
        "   Add GROQ_API_KEY to .env"
    )

# Load configuration
from context_engineering.config import (
    VECTOR_DIR, CACHE_DIR, TOP_K_RESULTS,
    CHAT_MODEL, EMBEDDING_MODEL, PROVIDER
)

provider = "GROQ"
print("Environment loaded")
print(f" Provider: {provider}")
print(f"Project root: {project_root}")

Environment loaded
 Provider: GROQ
Project root: c:\Users\viraj\Zuu\Real_State_Agent


### RAG

In [5]:
#Import Chat Services
from context_engineering.application.chat_service.rag_service import RAGService;

print(" Chat services loaded from service layer")

 Chat services loaded from service layer


In [ ]:
#Connect to Vector Store & Initialize LLM
from langchain_qdrant import Qdrant
from qdrant_client import QdrantClient
from context_engineering.infrastructure.llm_providers import (
    get_default_embeddings,
    get_chat_llm
)

# Use Groq for chat to reduce OpenRouter usage
llm = get_chat_llm(temperature=0, provider="groq")

# Embeddings require an OpenAI-compatible provider
embedding_provider = "openrouter"
embeddings = get_default_embeddings(provider=embedding_provider)

# Keep query embeddings short to reduce token usage
MAX_QUERY_CHARS = 600
def _compact_query(text: str) -> str:
    normalized = " ".join((text or "").split())
    if len(normalized) > MAX_QUERY_CHARS:
        return normalized[:MAX_QUERY_CHARS].rstrip()
    return normalized

print(f"LLM initialized: {CHAT_MODEL}")
print(f"Embeddings initialized: {EMBEDDING_MODEL} ({embedding_provider})")
print(f"Provider: {PROVIDER}")

# Connect to Qdrant local store
qdrant_path = VECTOR_DIR / "qdrant"
if not qdrant_path.exists():
    raise FileNotFoundError(" Run 02_chunking.ipynb first")

# Close any existing client to avoid local lock errors
if "qdrant_client" in globals():
    try:
        qdrant_client.close()
    except Exception:
        pass

# qdrant-client 1.17+ removed search/search_points; map to query_points
if not hasattr(QdrantClient, "search") and hasattr(QdrantClient, "query_points"):
    def _search_compat(
        self,
        collection_name,
        query_vector,
        limit=10,
        offset=None,
        filter=None,
        params=None,
        with_payload=True,
        with_vectors=False,
        score_threshold=None,
        consistency=None,
        shard_key_selector=None,
        timeout=None,
        **kwargs,
    ):
        # Avoid duplicate kwargs that may already be provided by upstream callers
        kwargs.pop("query_filter", None)
        kwargs.pop("search_params", None)
        kwargs.pop("limit", None)
        kwargs.pop("offset", None)
        kwargs.pop("with_payload", None)
        kwargs.pop("with_vectors", None)
        kwargs.pop("score_threshold", None)
        result = self.query_points(
            collection_name=collection_name,
            query=query_vector,
            using=None,
            prefetch=None,
            query_filter=filter,
            search_params=params,
            limit=limit,
            offset=offset,
            with_payload=with_payload,
            with_vectors=with_vectors,
            score_threshold=score_threshold,
            lookup_from=None,
            consistency=consistency,
            shard_key_selector=shard_key_selector,
            timeout=timeout,
            **kwargs,
        )
        return result.points

    QdrantClient.search = _search_compat

qdrant_client = QdrantClient(path=str(qdrant_path))
vectorstore = Qdrant(
    client=qdrant_client,
    collection_name="fixed",
    embeddings=embeddings
 )

# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K_RESULTS}
 )

print("✅ Connected to vector store")
print(f"📊 Collection size: {vectorstore.client.count(collection_name='fixed').count} vectors")

LLM initialized: llama-3.1-8b-instant
Embeddings initialized: openai/text-embedding-3-large (openrouter)
Provider: groq
✅ Connected to vector store
📊 Collection size: 464 vectors


In [28]:
# Initialize RAG Service
rag_service = RAGService(
    retriever=retriever,
    llm=llm,
    k=TOP_K_RESULTS
)

print("RAGService initialized")
print(f" Retrieval: top-{TOP_K_RESULTS} documents")

RAGService initialized
 Retrieval: top-4 documents


In [ ]:
query = _compact_query("Colombo")
docs = retriever.invoke(query)
len(docs), docs[0].page_content[:500], docs[0].metadata

APIStatusError: Error code: 402 - {'error': {'message': 'Insufficient credits. This account never purchased credits. Make sure your key is on the correct account or org, and if so, purchase more at https://openrouter.ai/settings/credits', 'code': 402}}

In [ ]:
# Generate Answer with RAG Service
USER_QUERY = "What is Primelands and what types of properties are listed"
QUERY = _compact_query(USER_QUERY)

print(f"Query: {QUERY}\n")
print("=" * 80)
print("GENERATING ANSWER WITH RAG SERVICE...")
print("=" * 80)

result = rag_service.generate(QUERY)

print(f"\n⏱  Generation time: {result['generation_time']:.2f}s")
print(f" Documents used: {result['num_docs']}")
print("\n" + "=" * 80)
print("ANSWER")
print("=" * 80)
print(result['answer'])
print("\n" + "=" * 80)
print("EVIDENCE URLS")
print("=" * 80)
for url in result['evidence_urls']:
    print(f"  - {url}")

Query: What is Primelands and what types of properties are listed

GENERATING ANSWER WITH RAG SERVICE...

⏱  Generation time: 1.77s
 Documents used: 4

ANSWER
I don't have enough information to answer the user's question as there are no sources provided that contain information about Primelands.

EVIDENCE URLS


### CAG

In [11]:
#Import Chat Services
from context_engineering.application.chat_service.cag_service import CAGService;
from context_engineering.application.chat_service.cag_cache import CAGCache;

print(" CAG services loaded from service layer")

 CAG services loaded from service layer


In [20]:
# Cell 7: Initialize CAG Service with Semantic Cache
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Create semantic cache (embedder required for similarity matching)
cache = CAGCache(
    cache_dir=CACHE_DIR,
    embedder=embeddings,  # Uses same embeddings as vector store
    similarity_threshold=0.90,  # Catches paraphrased questions
    history_ttl_hours=24  # History expires after 24 hours
)

# Create CAG service
cag_service = CAGService(
    rag_service=rag_service,
    cache=cache
)

print("CAGService initialized (semantic matching)")
print(f" Cache directory: {CACHE_DIR}")
stats = cache.stats()
print(f"Cached responses: {stats['total_cached']}")
print(f"Similarity threshold: {stats['similarity_threshold']}")
print(f"History TTL: {stats['history_ttl_hours']}h")

CAGService initialized (semantic matching)
 Cache directory: c:\Users\viraj\Zuu\Real_State_Agent\data\cag_cache
Cached responses: 0
Similarity threshold: 0.9
History TTL: 24h


In [26]:
# FAQs are defined in config/faqs.yaml

from context_engineering.config import KNOWN_FAQS

print(f"Found {len(KNOWN_FAQS)} FAQs in config/faqs.yaml")
print("\n Sample FAQs:")
for faq in KNOWN_FAQS[:5]:
    print(f"   - {faq}")
print(f"   ... and {len(KNOWN_FAQS) - 5} more\n")

# Load FAQs into cache (this just registers them, doesn't generate responses yet)
loaded = cag_service.load_faqs(KNOWN_FAQS)
print(f"✅ Loaded {loaded} new FAQs into cache")

# To warm FAQs (generate responses), uncomment:
# print("\n🔥 Warming FAQs (this may take a few minutes)...")
# cag_service.warm_faqs()

print("\n💡 Tip: Run cag_service.warm_faqs() to pre-generate FAQ responses")

Found 27 FAQs in config/faqs.yaml

 Sample FAQs:
   - What is Primelands and what types of properties are listed?
   - How do I search for land, apartments, houses, or portfolio properties?
   - Can I filter properties by location, budget, or property type?
   - Are the listings updated regularly?
   - Where can I find the price, size, and key features of a property?
   ... and 22 more



BadRequestError: Error code: 400 - {'error': {'message': 'invalid model ID', 'type': 'invalid_request_error', 'param': None, 'code': None}}

### CRAG